# Google Colab: smoke test обучения

Ноутбук использует текущее окружение Colab без создания отдельного virtualenv.
Базовый тестовый сценарий: `ScanObjectNN + DGCNN + TensorBoard`.

In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/magicnet-cloud")
DATA_DIR = Path("/content/data")
LOG_ROOT = Path("/content/magicnet-cloud/log")
CONFIG_PATH = "cfgs/scanobjectnn/dgcnn.yaml"

TASK_RUNNERS = {
    "scanobjectnn": "examples/classification/main.py",
    "scanobjectnn_pix4point": "examples/classification/main.py",
    "modelnet40ply2048": "examples/classification/main.py",
    "s3dis": "examples/segmentation/main.py",
    "s3dis_pix4point": "examples/segmentation/main.py",
    "scannet": "examples/segmentation/main.py",
    "k3d_xyz": "examples/segmentation/main.py",
    "k3d_xyzrgb": "examples/segmentation/main.py",
    "shapenetpart": "examples/shapenetpart/main.py",
    "shapenetpart_pix4point": "examples/shapenetpart/main.py",
}

TASK_NAME = Path(CONFIG_PATH).parts[1]
ENTRYPOINT = TASK_RUNNERS.get(TASK_NAME, "examples/classification/main.py")

EXTRA_OPTS = [
    f"data_dir={DATA_DIR}",
    f"log_root={LOG_ROOT}",
    "epochs=1",
    "batch_size=8",
    "val_batch_size=8",
    "dataloader.num_workers=2",
    "wandb.use_wandb=False",
]

REPO_DIR, DATA_DIR, LOG_ROOT, CONFIG_PATH, ENTRYPOINT

Если датасеты и логи лежат на Google Drive, сначала смонтируйте диск и поменяйте `DATA_DIR` и `LOG_ROOT`, например на `/content/drive/MyDrive/data` и `/content/drive/MyDrive/pointnext_logs`.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os

if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"Репозиторий не найден: {REPO_DIR}. Склонируйте его в Colab или поменяйте REPO_DIR."
    )

os.chdir(REPO_DIR)
print(Path.cwd())

Скрипт ниже ставит только недостающие зависимости в текущее окружение Colab.

Если позже понадобится PointNeXt / PointNet++, можно до запуска выставить:
`INSTALL_POINTNET2_BATCH=1`.
Для конфигов с `torch_scatter`: `INSTALL_TORCH_SCATTER=1`.

Ноутбук сам выбирает runner по `CONFIG_PATH`: classification / segmentation / shapenetpart.

In [ ]:
import os
import subprocess

env = os.environ.copy()
env.setdefault("INSTALL_TORCH_SCATTER", "0")
env.setdefault("INSTALL_POINTNET2_BATCH", "0")

cmd = ["bash", "script/install_colab_requirements.sh"]
print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)

In [ ]:
import shlex
import subprocess

cmd = [
    "python",
    ENTRYPOINT,
    "--cfg",
    CONFIG_PATH,
    *EXTRA_OPTS,
]

print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path

task_name = Path(CONFIG_PATH).parts[1]
log_root = LOG_ROOT / task_name
run_dirs = sorted(
    [path for path in log_root.iterdir() if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
)

if not run_dirs:
    raise RuntimeError(f"Не найдено ни одного run-директория в {log_root}")

latest_run = run_dirs[-1]
logdir = str(latest_run)
print(f"TensorBoard logdir: {logdir}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $logdir